# 06 — Shallow + Deep Learning for Sentiment Analysis
**Covers:** Word embeddings · CNN · LSTM · GRU · BiLSTM · Attention · Comparison

---

In [ ]:
!pip install -q torch torchtext datasets scikit-learn matplotlib pandas numpy

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score
import re

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

%matplotlib inline

## 1. Load & Tokenise Data

In [ ]:
from datasets import load_dataset

ds = load_dataset("glue", "sst2")
train_texts = ds['train']['sentence']
train_labels = ds['train']['label']
val_texts  = ds['validation']['sentence']
val_labels = ds['validation']['label']

def simple_tokenize(text):
    return re.sub(r'[^a-z0-9\s]', '', text.lower()).split()

# Build vocabulary
MAX_VOCAB = 20_000
MAX_LEN   = 64
PAD_IDX, UNK_IDX = 0, 1

counter = Counter()
for t in train_texts:
    counter.update(simple_tokenize(t))

vocab = {word: idx+2 for idx, (word, _) in enumerate(counter.most_common(MAX_VOCAB))}
vocab['<PAD>'] = PAD_IDX
vocab['<UNK>'] = UNK_IDX
VOCAB_SIZE = len(vocab)
print(f"Vocabulary size: {VOCAB_SIZE}")

def encode(text, max_len=MAX_LEN):
    tokens = simple_tokenize(text)
    ids = [vocab.get(t, UNK_IDX) for t in tokens[:max_len]]
    ids += [PAD_IDX] * (max_len - len(ids))
    return ids

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.X = torch.tensor([encode(t) for t in texts], dtype=torch.long)
        self.y = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_ds = SentimentDataset(train_texts, train_labels)
val_ds   = SentimentDataset(val_texts,   val_labels)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=128)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 2. Model Definitions

In [ ]:
EMB_DIM = 128
HIDDEN  = 128
N_CLASSES = 2

# --- 2a. CNN (shallow deep) ---
class TextCNN(nn.Module):
    def __init__(self, vocab_size, emb_dim, n_filters=64, filter_sizes=(2,3,4), n_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.convs = nn.ModuleList([
            nn.Conv2d(1, n_filters, (fs, emb_dim)) for fs in filter_sizes
        ])
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(n_filters * len(filter_sizes), n_classes)

    def forward(self, x):
        x = self.embedding(x).unsqueeze(1)           # (B, 1, L, E)
        x = [torch.relu(c(x)).squeeze(3) for c in self.convs]  # [(B, F, L-fs+1)]
        x = [torch.max(t, dim=2).values for t in x]  # [(B, F)]
        x = torch.cat(x, dim=1)                       # (B, F*len_filters)
        return self.fc(self.dropout(x))

# --- 2b. BiLSTM ---
class BiLSTMSentiment(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden, n_classes=2, n_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(emb_dim, hidden, num_layers=n_layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * 2, n_classes)

    def forward(self, x):
        x = self.dropout(self.embedding(x))
        out, (h, _) = self.lstm(x)
        h = torch.cat([h[-2], h[-1]], dim=1)  # last fwd + bwd hidden
        return self.fc(self.dropout(h))

# --- 2c. GRU with Attention ---
class Attention(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.w = nn.Linear(hidden * 2, 1)
    def forward(self, enc_out):  # (B, L, 2H)
        scores = self.w(enc_out).squeeze(-1)      # (B, L)
        weights = torch.softmax(scores, dim=1)
        context = (enc_out * weights.unsqueeze(-1)).sum(dim=1)  # (B, 2H)
        return context, weights

class GRUAttentionSentiment(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden, n_classes=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.gru = nn.GRU(emb_dim, hidden, batch_first=True, bidirectional=True, num_layers=2, dropout=dropout)
        self.attention = Attention(hidden)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * 2, n_classes)

    def forward(self, x):
        x = self.dropout(self.embedding(x))
        out, _ = self.gru(x)
        context, _ = self.attention(out)
        return self.fc(self.dropout(context))

## 3. Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, preds, trues = 0, [], []
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        preds.extend(out.argmax(1).cpu().tolist())
        trues.extend(y.cpu().tolist())
    return total_loss / len(loader), accuracy_score(trues, preds)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, preds, trues = 0, [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            out = model(X)
            total_loss += criterion(out, y).item()
            preds.extend(out.argmax(1).cpu().tolist())
            trues.extend(y.cpu().tolist())
    return total_loss / len(loader), accuracy_score(trues, preds), f1_score(trues, preds)

def run_training(model_class, model_kwargs, epochs=5, lr=1e-3, model_name='Model'):
    model = model_class(**model_kwargs).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=1, factor=0.5)
    criterion = nn.CrossEntropyLoss()
    
    history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[], 'val_f1':[]}
    best_f1, best_state = 0, None
    
    for epoch in range(1, epochs+1):
        tl, ta = train_epoch(model, train_loader, optimizer, criterion)
        vl, va, vf = evaluate(model, val_loader, criterion)
        scheduler.step(vl)
        if vf > best_f1:
            best_f1 = vf
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        history['train_loss'].append(tl); history['train_acc'].append(ta)
        history['val_loss'].append(vl);   history['val_acc'].append(va); history['val_f1'].append(vf)
        print(f"[{model_name}] Epoch {epoch}/{epochs} | "
              f"Train Loss={tl:.4f} Acc={ta:.4f} | Val Loss={vl:.4f} Acc={va:.4f} F1={vf:.4f}")
    
    model.load_state_dict(best_state)
    return model, history

## 4. Train All Models

In [ ]:
EPOCHS = 5
results = {}

# TextCNN
cnn_model, cnn_hist = run_training(
    TextCNN,
    dict(vocab_size=VOCAB_SIZE, emb_dim=EMB_DIM, n_filters=64, filter_sizes=(2,3,4)),
    epochs=EPOCHS, model_name='TextCNN'
)
results['TextCNN'] = cnn_hist

In [ ]:
# BiLSTM
lstm_model, lstm_hist = run_training(
    BiLSTMSentiment,
    dict(vocab_size=VOCAB_SIZE, emb_dim=EMB_DIM, hidden=HIDDEN),
    epochs=EPOCHS, model_name='BiLSTM'
)
results['BiLSTM'] = lstm_hist

In [ ]:
# GRU + Attention
gru_model, gru_hist = run_training(
    GRUAttentionSentiment,
    dict(vocab_size=VOCAB_SIZE, emb_dim=EMB_DIM, hidden=HIDDEN),
    epochs=EPOCHS, model_name='GRU+Attention'
)
results['GRU+Attention'] = gru_hist

## 5. Training Curves & Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = {'TextCNN': '#1E88E5', 'BiLSTM': '#43A047', 'GRU+Attention': '#E53935'}

for name, hist in results.items():
    axes[0].plot(hist['train_loss'], label=name, color=colors[name])
    axes[1].plot(hist['val_acc'],   label=name, color=colors[name])
    axes[2].plot(hist['val_f1'],    label=name, color=colors[name])

axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].set_title('Val Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[2].set_title('Val F1'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.suptitle('Training Curves — Shallow/Deep Models', fontsize=13)
plt.tight_layout()
plt.show()

# Final comparison table
comp = pd.DataFrame({
    'Model': list(results.keys()),
    'Best Val Acc': [max(h['val_acc']) for h in results.values()],
    'Best Val F1':  [max(h['val_f1'])  for h in results.values()],
}).sort_values('Best Val F1', ascending=False)
print(comp.to_string(index=False))

## 6. Attention Visualisation (GRU+Attention)

In [ ]:
def visualise_attention(text, model, vocab, device=DEVICE):
    model.eval()
    tokens = simple_tokenize(text)
    ids = [vocab.get(t, UNK_IDX) for t in tokens[:MAX_LEN]]
    X = torch.tensor([ids + [PAD_IDX]*(MAX_LEN-len(ids))], dtype=torch.long).to(device)
    
    with torch.no_grad():
        emb = model.dropout(model.embedding(X))
        out, _ = model.gru(emb)
        _, weights = model.attention(out)
    
    weights = weights.squeeze().cpu().numpy()[:len(tokens)]
    
    fig, ax = plt.subplots(figsize=(max(8, len(tokens)*0.6), 1.5))
    ax.bar(range(len(tokens)), weights, color=plt.cm.RdYlGn(weights / weights.max()))
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=10)
    ax.set_title(f'Attention weights: "{text[:50]}"', fontsize=11)
    ax.set_ylabel('Weight')
    plt.tight_layout()
    plt.show()

visualise_attention("This film is absolutely stunning and deeply moving", gru_model, vocab)
visualise_attention("Dull, lifeless, and a complete waste of time", gru_model, vocab)

---
**Next Notebook →** `07_deep_learning.ipynb`